In [2]:
# ==========================================================
# Notebook 05: Skill Gap Analysis
# ==========================================================

import re
import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
sections_df = pd.read_csv("processed/resume_sections.csv")

sections_df.head()

,file_name,skills,experience,education,certifications
0,10554236.pdf,skills accounting; general accounting; account...,experience company name july 2011 to november ...,education northern maine community college 199...,certifications certified defense financial man...
1,10674770.pdf,skills. highlights dba quick books mas - sage ...,experience staff accountant january 2014 to oc...,"education bachelor of science : accounting , m...",NaN
2,11163645.pdf,skills and attributes. attributes self-motivat...,experience accountant january 2011 to november...,education computer applications specialist cer...,NaN
3,11759079.pdf,skills by drafting over forty memorandums that...,experience company name june 2011 to current s...,"education emory university, goizueta business ...",certifications and awards fulton county casa b...
4,12065211.pdf,skills aderant/cms financial reporting excel u...,experience in full life cycle of general ledge...,education bachelor of business administration ...,"certifications certified public accountant, ne..."


In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("SBERT model loaded.")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


SBERT model loaded.


In [5]:
job_description = {
    "skills": """
    Financial Accounting
    Bookkeeping
    Accounts Payable
    Accounts Receivable
    General Ledger
    Bank Reconciliation
    Financial Reporting
    Taxation
    GST
    TDS
    Payroll Processing
    Auditing
    Budgeting
    Microsoft Excel
    Tally ERP
    SAP FICO
    QuickBooks
    ERP Systems
    """,
    "experience": """
    Minimum 3 years experience
    in accounting, bookkeeping,
    financial reporting, taxation,
    and ERP-based financial systems.
    Experience with Tally, SAP,
    or QuickBooks is preferred.
    """,
    "education": """
    Bachelor's or Master's degree
    in Commerce, Accounting,
    Finance, or a related field.
    Professional certifications
    such as CA, CPA, CMA, or ACCA
    are an added advantage.
    """,
}

In [6]:
SKILL_VOCABULARY = [
    "accounting",
    "financial accounting",
    "bookkeeping",
    "accounts payable",
    "accounts receivable",
    "general ledger",
    "journal entries",
    "bank reconciliation",
    "financial reporting",
    "financial statements",
    "balance sheet",
    "income statement",
    "cash flow statement",
    "taxation",
    "gst",
    "tds",
    "income tax",
    "payroll processing",
    "auditing",
    "internal audit",
    "external audit",
    "budgeting",
    "financial analysis",
    "cost accounting",
    "management accounting",
    "invoice processing",
    "account reconciliation",
    "fixed assets accounting",
    "accounts finalization",
    "compliance",
    "ms excel",
    "microsoft excel",
    "advanced excel",
    "tally",
    "tally erp",
    "tally prime",
    "sap",
    "sap fico",
    "quickbooks",
    "oracle financials",
    "erp systems",
    "zoho books",
    "xero",
    "financial modeling",
    "variance analysis",
    "cash management",
    "vendor management",
    "procurement accounting",
    "data entry",
    "mis reporting",
]

In [9]:
def extract_skills(text, skill_vocab):

    # If input is a dictionary, combine all values
    if isinstance(text, dict):
        text = " ".join(text.values())

    text = text.lower()

    extracted = []

    for skill in skill_vocab:
        if skill.lower() in text:
            extracted.append(skill)

    return sorted(list(set(extracted)))

In [10]:
jd_skills = extract_skills(job_description, SKILL_VOCABULARY)

jd_skills

['accounting',
 'accounts payable',
 'accounts receivable',
 'auditing',
 'bank reconciliation',
 'bookkeeping',
 'budgeting',
 'erp systems',
 'financial accounting',
 'financial reporting',
 'general ledger',
 'gst',
 'microsoft excel',
 'payroll processing',
 'quickbooks',
 'sap',
 'sap fico',
 'tally',
 'tally erp',
 'taxation',
 'tds']

In [11]:
candidate_skills = extract_skills(sections_df.loc[0, "skills"], SKILL_VOCABULARY)

candidate_skills

['accounting', 'accounts payable']

In [12]:
matched_skills = list(set(jd_skills).intersection(set(candidate_skills)))

missing_skills = list(set(jd_skills).difference(set(candidate_skills)))

print("Matched:", matched_skills)

print("Missing:", missing_skills)

Matched: ['accounting', 'accounts payable']
Missing: ['auditing', 'sap', 'budgeting', 'tally erp', 'erp systems', 'taxation', 'accounts receivable', 'quickbooks', 'financial accounting', 'payroll processing', 'financial reporting', 'gst', 'sap fico', 'bank reconciliation', 'general ledger', 'bookkeeping', 'tally', 'tds', 'microsoft excel']


In [13]:
def semantic_skill_similarity(skill_a, skill_b, model):

    emb_a = model.encode(skill_a)

    emb_b = model.encode(skill_b)

    score = cosine_similarity([emb_a], [emb_b])[0][0]

    return score

In [14]:
semantic_skill_similarity("deep learning", "neural networks", model)

0.75467837

In [15]:
def find_semantic_matches(jd_skills, resume_skills, model, threshold=0.75):

    semantic_matches = []

    unresolved = []

    for jd_skill in jd_skills:

        best_score = 0

        best_match = None

        for resume_skill in resume_skills:

            score = semantic_skill_similarity(jd_skill, resume_skill, model)

            if score > best_score:

                best_score = score

                best_match = resume_skill

        if best_score >= threshold:

            semantic_matches.append(
                {
                    "jd_skill": jd_skill,
                    "matched_with": best_match,
                    "similarity": round(best_score, 3),
                }
            )

        else:

            unresolved.append(jd_skill)

    return semantic_matches, unresolved

In [16]:
semantic_matches, remaining_gaps = find_semantic_matches(
    jd_skills, candidate_skills, model
)

semantic_matches

[{'jd_skill': 'accounting', 'matched_with': 'accounting', 'similarity': 1.0},
 {'jd_skill': 'accounts payable',
  'matched_with': 'accounts payable',
  'similarity': 1.0},
 {'jd_skill': 'financial accounting',
  'matched_with': 'accounting',
  'similarity': 0.889}]

In [17]:
def generate_gap_report(jd_skills, candidate_skills, semantic_matches, remaining_gaps):

    report = {
        "required_skills": len(jd_skills),
        "matched_skills": len(semantic_matches),
        "missing_skills": remaining_gaps,
        "coverage_percent": round(100 * len(semantic_matches) / len(jd_skills), 2),
    }

    return report

In [18]:
gap_report = generate_gap_report(
    jd_skills, candidate_skills, semantic_matches, remaining_gaps
)

gap_report

{'required_skills': 21,
 'matched_skills': 3,
 'missing_skills': ['accounts receivable',
  'auditing',
  'bank reconciliation',
  'bookkeeping',
  'budgeting',
  'erp systems',
  'financial reporting',
  'general ledger',
  'gst',
  'microsoft excel',
  'payroll processing',
  'quickbooks',
  'sap',
  'sap fico',
  'tally',
  'tally erp',
  'taxation',
  'tds'],
 'coverage_percent': 14.29}

In [19]:
skill_matrix = []

for skill in jd_skills:

    if skill in candidate_skills:

        status = "✅ Present"

    else:

        status = "❌ Missing"

    skill_matrix.append({"Skill": skill, "Status": status})

skill_matrix_df = pd.DataFrame(skill_matrix)

skill_matrix_df

,Skill,Status
0,accounting,✅ Present
1,accounts payable,✅ Present
2,accounts receivable,❌ Missing
3,auditing,❌ Missing
4,bank reconciliation,❌ Missing
5,bookkeeping,❌ Missing
6,budgeting,❌ Missing
7,erp systems,❌ Missing
8,financial accounting,❌ Missing
9,financial reporting,❌ Missing


In [20]:
if len(remaining_gaps) > 0:

    recommendation = "Candidate can improve " "profile by learning: " + ", ".join(
        remaining_gaps
    )

else:

    recommendation = "No major skill gaps detected."

print(recommendation)

Candidate can improve profile by learning: accounts receivable, auditing, bank reconciliation, bookkeeping, budgeting, erp systems, financial reporting, general ledger, gst, microsoft excel, payroll processing, quickbooks, sap, sap fico, tally, tally erp, taxation, tds
